# Clase de cierre: Del dato al hallazgo

**Duración:** 60 minutos  
**Objetivo:** integrar limpieza, transformación, agrupación y visualización con Pandas y Matplotlib.

## 1. Repaso del flujo

Durante el análisis seguiremos este recorrido:

```text
CSV -> inspección -> limpieza -> transformación -> agrupación -> visualización -> conclusión
```

## 2. Carga e inspección

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams["font.size"] = 11

In [ ]:
ruta_csv = Path("../clase_11/recursos/ventas_ecommerce.csv")

if not ruta_csv.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {ruta_csv}")

df_original = pd.read_csv(ruta_csv)

print(f"Filas y columnas: {df_original.shape}")
display(df_original.head())
display(df_original.dtypes)

Identifica:

- qué columnas son numéricas;
- qué columnas contienen texto o categorías;
- qué columna representa fechas;
- qué variables puedes usar para agrupar.

## 3. Limpieza y creación de la métrica

In [ ]:
print("Valores faltantes por columna:")
display(df_original.isna().sum())

print(f"Filas duplicadas: {df_original.duplicated().sum()}")

In [ ]:
# Conservamos filas completas y eliminamos duplicados para este análisis.
df_ventas = (
    df_original
    .dropna()
    .drop_duplicates()
    .copy()
)

df_ventas["Fecha"] = pd.to_datetime(
    df_ventas["Fecha"],
    format="%Y-%m-%d",
)
df_ventas["monto_total"] = (
    df_ventas["Precio_Unitario"] * df_ventas["Cantidad"]
)

print(f"Filas originales: {len(df_original)}")
print(f"Filas listas para analizar: {len(df_ventas)}")
display(
    df_ventas[
        ["Fecha", "Categoria", "Precio_Unitario", "Cantidad", "monto_total"]
    ].head()
)

## 4. Ventas por fecha

In [ ]:
ventas_por_fecha = (
    df_ventas
    .groupby("Fecha")["monto_total"]
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    ventas_por_fecha.index,
    ventas_por_fecha.values,
    color="royalblue",
    marker="o",
    linewidth=1.5,
)
ax.set_title("Monto total de ventas por fecha")
ax.set_xlabel("Fecha")
ax.set_ylabel("Monto total (MXN)")
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
fecha_maxima = ventas_por_fecha.idxmax()
monto_maximo = ventas_por_fecha.max()

print(f"Mayor venta: {fecha_maxima:%Y-%m-%d} con ${monto_maximo:,.2f} MXN")

### Pregunta

¿En qué fecha ocurre el pico de ventas y qué lectura de negocio puedes hacer?

## 5. Ventas por categoría

In [ ]:
ventas_por_categoria = (
    df_ventas
    .groupby("Categoria")["monto_total"]
    .sum()
    .sort_values(ascending=False)
)

colores = {
    categoria: color
    for categoria, color in zip(
        ventas_por_categoria.index,
        ["teal", "darkorange", "seagreen", "mediumpurple", "crimson"],
    )
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    ventas_por_categoria.index,
    ventas_por_categoria.values,
    color=[colores[categoria] for categoria in ventas_por_categoria.index],
)
ax.set_title("Monto total de ventas por categoría")
ax.set_xlabel("Categoría")
ax.set_ylabel("Monto total (MXN)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()

In [ ]:
categoria_maxima = ventas_por_categoria.idxmax()
monto_categoria_maxima = ventas_por_categoria.max()

print(
    f"La categoría con mayor monto fue {categoria_maxima}, "
    f"con ${monto_categoria_maxima:,.2f} MXN."
)

## 6. Distribución de precios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    df_ventas["Precio_Unitario"],
    bins=12,
    color="coral",
    edgecolor="black",
)
ax.set_title("Distribución de precios unitarios")
ax.set_xlabel("Precio unitario (MXN)")
ax.set_ylabel("Frecuencia")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 7. Mini-reporte final

Completa las siguientes respuestas usando los valores y gráficos observados:

In [ ]:
print(
    f"La categoría con mayor monto fue {categoria_maxima}, "
    f"con ${monto_categoria_maxima:,.2f} MXN."
)
print("Hallazgo sobre la fecha pico: ______________________________")
print("Hallazgo sobre la distribución de precios: __________________")
print("Decisión de negocio posible: ________________________________")
print("Limitación del análisis: ____________________________________")

## Cierre

Recuerda el flujo completo:

```text
Inspeccionar -> limpiar -> transformar -> agrupar -> visualizar -> comunicar
```

Una conclusión válida debe apoyarse en datos observables y reconocer sus limitaciones.